In [5]:
import os
import sys

if sys.platform == 'win32' and sys.stdout.encoding.lower() != 'utf-8':
    sys.stdout.reconfigure(encoding='utf-8')

# Ensure we can import from src
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import dagshub
import mlflow
from statsmodels.graphics.tsaplots import plot_acf
from torch.utils.data import DataLoader

from models.ssm import LatentSSM
from feature_prep import SSMDataset

ModuleNotFoundError: No module named 'torch'

In [ ]:
print("--- Connecting to DagsHub MLflow ---")
os.environ["MLFLOW_TRACKING_USERNAME"] = "rgorasia19"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "26b84ad06cab16667533b56a5cebe4ebf27d8f9a"

dagshub.init(repo_owner="rgorasia19", repo_name="Energy-Intelligence-System", mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/rgorasia19/Energy-Intelligence-System.mlflow")

experiment = mlflow.get_experiment_by_name("hssm_probabilistic")
if experiment is None:
    raise ValueError("Experiment 'hssm_probabilistic' not found.")
    
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["start_time desc"], max_results=1)
if runs.empty:
    raise ValueError("No runs found in experiment 'hssm_probabilistic'.")
    
run_id = runs.iloc[0].run_id
print(f"Latest Run ID: {run_id}")

print("--- Downloading Model and Artifacts ---")
local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="")
model_path = os.path.join(local_dir, "ssm_v1.pth")

data_dir = '../../../datalake/ssm_tensors/'
col_info = joblib.load(os.path.join(data_dir, 'columns.pkl'))
feature_cols = col_info['feature_columns']
target_cols = col_info['target_columns']

demand_cols = ['ND', 'TSD', 'ENGLAND_WALES_DEMAND']
gen_cols = ['GAS', 'COAL', 'NUCLEAR', 'WIND', 'GENERATION']

demand_idx = [target_cols.index(c) for c in demand_cols]
gen_idx = [target_cols.index(c) for c in gen_cols]

seq_len = 60
horizon = 30
latent_dim = 8
hidden_dim = 64
num_regimes = 4

weather_cols = ['temperature_2m', 'cloudcover', 'windspeed_10m', 'shortwave_radiation']
calendar_cols = [c for c in feature_cols if c.endswith('_sin') or c.endswith('_cos') or c == 'is_bank_holiday']
embedded_cols = ['EMBEDDED_WIND_CAPACITY', 'EMBEDDED_SOLAR_CAPACITY']
macro_cols = ['uk_cpi', 'uk_gdp_index', 'bank_rate']
known_columns = weather_cols + calendar_cols + [c for c in embedded_cols + macro_cols if c in feature_cols]
known_dim = len(known_columns)

model = LatentSSM(
    input_dim=len(feature_cols),
    demand_dim=len(demand_cols),
    gen_dim=len(gen_cols),
    known_dim=known_dim,
    latent_dim=latent_dim,
    hidden_dim=hidden_dim,
    num_regimes=num_regimes
)
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

print("--- Loading Test Data ---")
test_data = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))

batch_size = 256
test_dataset = SSMDataset(test_data, seq_len=seq_len, horizon=horizon, 
                          feature_columns=feature_cols, target_columns=target_cols,
                          known_columns=known_columns)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Model loaded successfully.")


In [ ]:
all_d_mean = []
all_d_var = []
all_g_mean = []
all_g_var = []
all_d_true = []
all_g_true = []
all_mask = []
all_z_seq = []
all_r_seq = []
all_features = []

print("--- Running Inference ---")
with torch.no_grad():
    for batch in test_loader:
        enc_inputs = batch['encoder_inputs'].to(device)
        dec_inputs = batch['decoder_inputs'].to(device)
        dec_targets = batch['decoder_targets'].numpy()
        dec_masks = batch['decoder_mask'].numpy()
        
        N = 50
        batch_d_samples = []
        batch_g_samples = []
        for _ in range(N):
            outputs = model(enc_inputs, dec_inputs, horizon, target_seq=None, sample=True, tau=1.0)
            d_sample = np.random.normal(loc=outputs['demand_mean'].cpu().numpy(), scale=np.sqrt(outputs['demand_var'].cpu().numpy()))
            g_sample = np.random.normal(loc=outputs['gen_mean'].cpu().numpy(), scale=np.sqrt(outputs['gen_var'].cpu().numpy()))
            batch_d_samples.append(d_sample)
            batch_g_samples.append(g_sample)
        
        batch_d_samples = np.stack(batch_d_samples, axis=1)
        batch_g_samples = np.stack(batch_g_samples, axis=1)
        
        all_d_mean.append(np.mean(batch_d_samples, axis=1))
        all_d_var.append(np.var(batch_d_samples, axis=1))
        all_g_mean.append(np.mean(batch_g_samples, axis=1))
        all_g_var.append(np.var(batch_g_samples, axis=1))
        
        all_d_true.append(dec_targets[:, :, demand_idx])
        all_g_true.append(dec_targets[:, :, gen_idx])
        all_mask.append(dec_masks[:, :, demand_idx])
        
        all_z_seq.append(outputs['sampled_z_seq'].cpu().numpy())
        all_r_seq.append(outputs['sampled_r_seq'].cpu().numpy())
        all_features.append(dec_inputs[:, :, -known_dim:].cpu().numpy())
        
d_mean = np.concatenate(all_d_mean, axis=0)
d_var = np.concatenate(all_d_var, axis=0) + 1e-6
g_mean = np.concatenate(all_g_mean, axis=0)
g_var = np.concatenate(all_g_var, axis=0) + 1e-6
d_true = np.concatenate(all_d_true, axis=0)
g_true = np.concatenate(all_g_true, axis=0)
mask = np.concatenate(all_mask, axis=0)
z_seq = np.concatenate(all_z_seq, axis=0)
r_seq = np.concatenate(all_r_seq, axis=0)
features = np.concatenate(all_features, axis=0)

valid_mask = mask == 1
d0_mean_flat = d_mean[:, :, 0][valid_mask[:, :, 0]]
d0_var_flat = d_var[:, :, 0][valid_mask[:, :, 0]]
d0_true_flat = d_true[:, :, 0][valid_mask[:, :, 0]]
d0_error_flat = np.abs(d0_mean_flat - d0_true_flat)

g0_mean_flat = g_mean[:, :, 0][valid_mask[:, :, 0]]
g0_var_flat = g_var[:, :, 0][valid_mask[:, :, 0]]
g0_true_flat = g_true[:, :, 0][valid_mask[:, :, 0]]
g0_error_flat = np.abs(g0_mean_flat - g0_true_flat)

print("Inference completed.")


In [ ]:
# 1. Predicted Sigma vs Absolute Error
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(x=np.sqrt(d0_var_flat), y=d0_error_flat, alpha=0.1, ax=axes[0])
axes[0].set_xlabel('Predicted Sigma')
axes[0].set_ylabel('Absolute Error')
axes[0].set_title(f'Demand ({demand_cols[0]}): Sigma vs Abs Error')
axes[0].plot([0, max(np.sqrt(d0_var_flat))], [0, max(np.sqrt(d0_var_flat))], 'r--')

sns.scatterplot(x=np.sqrt(g0_var_flat), y=g0_error_flat, alpha=0.1, ax=axes[1])
axes[1].set_xlabel('Predicted Sigma')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title(f'Generation ({gen_cols[0]}): Sigma vs Abs Error')
axes[1].plot([0, max(np.sqrt(g0_var_flat))], [0, max(np.sqrt(g0_var_flat))], 'r--')

plt.tight_layout()
plt.show()


In [ ]:
# 2. Latent Trajectory Inspection
sample_idx = 0
sample_z = z_seq[sample_idx]
plt.figure(figsize=(12, 6))
for i in range(latent_dim):
    plt.plot(sample_z[:, i], label=f'z_{i}')
plt.title(f'Latent State Trajectory for Sample {sample_idx}')
plt.xlabel('Horizon Step')
plt.ylabel('Latent Value')
plt.legend()
plt.show()


In [ ]:
# 3. Autocorrelation of Residuals
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

d_resid_seq = d_mean[0, :, 0] - d_true[0, :, 0]
g_resid_seq = g_mean[0, :, 0] - g_true[0, :, 0]

plot_acf(d_resid_seq, ax=axes[0], title=f'Demand ({demand_cols[0]}): Residual ACF (Sample 0)', lags=horizon-1)
plot_acf(g_resid_seq, ax=axes[1], title=f'Generation ({gen_cols[0]}): Residual ACF (Sample 0)', lags=horizon-1)

plt.tight_layout()
plt.show()


In [ ]:
# 4. Regime Transition Matrix
r_labels = np.argmax(r_seq, axis=-1)
transitions = np.zeros((num_regimes, num_regimes))
for b in range(r_labels.shape[0]):
    for t in range(horizon - 1):
        curr_r = r_labels[b, t]
        next_r = r_labels[b, t+1]
        transitions[curr_r, next_r] += 1
        
row_sums = transitions.sum(axis=1, keepdims=True)
transition_probs = np.divide(transitions, row_sums, out=np.zeros_like(transitions), where=row_sums!=0)

plt.figure(figsize=(8, 6))
sns.heatmap(transition_probs, annot=True, cmap='Blues', fmt=".2f")
plt.title('Regime Transition Matrix')
plt.xlabel('Next Regime')
plt.ylabel('Current Regime')
plt.show()


In [ ]:
# 5. Error vs Exogenous Features
temp_idx = known_columns.index('temperature_2m') if 'temperature_2m' in known_columns else None

if temp_idx is not None:
    temp_flat = features[:, :, temp_idx][valid_mask[:, :, 0]]
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    sns.scatterplot(x=temp_flat, y=d0_error_flat, alpha=0.1, ax=axes[0])
    axes[0].set_xlabel('Scaled Temperature')
    axes[0].set_ylabel('Absolute Error')
    axes[0].set_title(f'Demand ({demand_cols[0]}): Error vs Temp')
    
    sns.scatterplot(x=temp_flat, y=g0_error_flat, alpha=0.1, ax=axes[1])
    axes[1].set_xlabel('Scaled Temperature')
    axes[1].set_ylabel('Absolute Error')
    axes[1].set_title(f'Generation ({gen_cols[0]}): Error vs Temp')
    
    plt.tight_layout()
    plt.show()
else:
    print("Feature 'temperature_2m' not found in known_columns.")
